# Mouse Skin Atlas - scVI Integration

This notebook integrates multiple mouse skin datasets using scVI for building a comprehensive skin atlas.

**Prerequisites**: Run `00_preprocess_geo_datasets.ipynb` and `00b_preprocess_ch25h_col5a1_trem2.ipynb` first.

## Internal Datasets (Pre-processed MTX):
1. **Burn/Sham** - Burn and sham wound mice (10x Flex, timepoints D10, D14, D19)
2. **LPCAT3** - Vav1Cre Lpcat3 fl/fl, Lpcat3 fl/fl, and WT normal skin mice (10x Flex)
3. **CH25H/COL5A1/TREM2** - CH25H KO, COL5A1 KO (CKO), TREM2 KO and WT controls (SoupX + scDblFinder pre-processed)

## GEO Datasets (Pre-processed via 00_preprocess_geo_datasets.ipynb):
4. **GSE142471 (Haensel)** - Epidermal basal cells in wound healing Day 4 (5 samples)
5. **GSE113854 (Guerrero-Juarez)** - Fibroblast heterogeneity in wound Day 12 (1 merged sample)
6. **GSE186527 (Mascharak)** - Scarring vs regenerative healing (9 timepoint samples)
7. **GSE218430 (Liu)** - LncRNA SNHG26 wound healing WT vs KO (4 samples)
8. **GSE178758 (Foster)** - Dermal fibroblast scRNA-seq Inner/Outer wound POD2/7/14 (11 samples)

All datasets use the same standardized format: `counts_*.mtx`, `gene_names_*.csv`, `metadata_*.csv`.

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from typing import Dict, List, Optional
import anndata as ad
from scipy.io import mmread
from scipy.sparse import csr_matrix

# scVI imports
import scvi
from scvi.model import SCVI

# Settings
warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

# Set random seed for reproducibility
scvi.settings.seed = 42

print(f"scanpy version: {sc.__version__}")
print(f"scvi version: {scvi.__version__}")

Seed set to 42


scanpy version: 1.11.5
scvi version: 1.4.1


## 1. Configuration - Add Your Datasets Here

To add a new dataset, simply add an entry to the `DATASET_CONFIG` dictionary.

### Required files per dataset:
- **counts_*.mtx**: Matrix Market file with raw counts (genes x cells)
- **metadata_*.csv**: Cell metadata with barcodes as first column
- **genes_*.csv** or **genes_*.txt**: Gene names (one per line or single column CSV)

In [ ]:
# Base path for data
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================================================
# DATASET CONFIGURATION
# ============================================================================
# All datasets use the same standardized format:
#   - counts_file: Path to MTX file (genes x cells format)
#   - metadata_file: Path to metadata CSV
#   - genes_file: Path to gene names file (one gene per line, no header)
#   - study_name: Short identifier for the study
#   - description: Description of the dataset
#   - cell_type_col: Column name for cell type annotations (None if not available)
#   - cell_type_detailed_col: Column name for detailed cell types (None if not available)
# ============================================================================

DATASET_CONFIG = {
    # --- Internal Datasets ---
    "burn_sham": {
        "counts_file": DATA_DIR / "counts_burn_sham.mtx",
        "metadata_file": DATA_DIR / "metadata_brun_sham.csv",
        "genes_file": DATA_DIR / "gene_names_burn_sham.csv",
        "study_name": "BurnSham",
        "description": "Burn and sham wound healing (D10, D14, D19)",
        "cell_type_col": "cell_types_simple_short",
        "cell_type_detailed_col": "cell_types_detailed",
    },
    "lpcat": {
        "counts_file": DATA_DIR / "counts_lpcat.mtx",
        "metadata_file": DATA_DIR / "metadata_lpcat.csv",
        "genes_file": DATA_DIR / "gene_names_lpcat.csv",
        "study_name": "LPCAT3",
        "description": "LPCAT3 KO/WT and normal skin",
        "cell_type_col": "cell_types_simple",
        "cell_type_detailed_col": "cell_types_detailed",
    },
    "ch25h_col5a1_trem2": {
        "counts_file": DATA_DIR / "counts_ch25h_col5a1_trem2.mtx",
        "metadata_file": DATA_DIR / "metadata_ch25h_col5a1_trem2.csv",
        "genes_file": DATA_DIR / "gene_names_ch25h_col5a1_trem2.csv",
        "study_name": "CH25H_COL5A1_TREM2",
        "description": "CH25H KO, COL5A1 KO (CKO), TREM2 KO and WT controls (SoupX + scDblFinder)",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
    # --- GEO Datasets (generated by 00_preprocess_geo_datasets.ipynb) ---
    "haensel_wound_d4": {
        "counts_file": DATA_DIR / "counts_haensel_wound_d4.mtx",
        "metadata_file": DATA_DIR / "metadata_haensel_wound_d4.csv",
        "genes_file": DATA_DIR / "gene_names_haensel_wound_d4.csv",
        "study_name": "Haensel_WoundD4",
        "description": "GSE142471 - Epidermal basal cells wound healing Day 4",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
    "guerrero_wound_d12": {
        "counts_file": DATA_DIR / "counts_guerrero_wound_d12.mtx",
        "metadata_file": DATA_DIR / "metadata_guerrero_wound_d12.csv",
        "genes_file": DATA_DIR / "gene_names_guerrero_wound_d12.csv",
        "study_name": "Guerrero_WoundD12",
        "description": "GSE113854 - Fibroblast heterogeneity wound Day 12",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
    "mascharak_regeneration": {
        "counts_file": DATA_DIR / "counts_mascharak_regeneration.mtx",
        "metadata_file": DATA_DIR / "metadata_mascharak_regeneration.csv",
        "genes_file": DATA_DIR / "gene_names_mascharak_regeneration.csv",
        "study_name": "Mascharak_Regen",
        "description": "GSE186527 - Scarring vs regenerative healing (YAP inhibition)",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
    "liu_snhg26": {
        "counts_file": DATA_DIR / "counts_liu_snhg26.mtx",
        "metadata_file": DATA_DIR / "metadata_liu_snhg26.csv",
        "genes_file": DATA_DIR / "gene_names_liu_snhg26.csv",
        "study_name": "Liu_SNHG26",
        "description": "GSE218430 - LncRNA SNHG26 wound healing WT vs KO",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
    "foster_scrna": {
        "counts_file": DATA_DIR / "counts_foster_scrna.mtx",
        "metadata_file": DATA_DIR / "metadata_foster_scrna.csv",
        "genes_file": DATA_DIR / "gene_names_foster_scrna.csv",
        "study_name": "Foster_Fibroblast",
        "description": "GSE178758 - Dermal fibroblast scRNA-seq (Inner/Outer POD2/7/14)",
        "cell_type_col": None,
        "cell_type_detailed_col": None,
    },
}

# ============================================================================
# INCLUDE/EXCLUDE DATASETS
# ============================================================================
# Set to True/False to include/exclude any dataset from the integration

INCLUDE_DATASETS = {
    # Internal
    "burn_sham": True,
    "lpcat": True,
    "ch25h_col5a1_trem2": True,
    # GEO
    "haensel_wound_d4": True,
    "guerrero_wound_d12": True,
    "mascharak_regeneration": True,
    "liu_snhg26": True,
    "foster_scrna": True,
}

print("Datasets configured:", list(DATASET_CONFIG.keys()))
print("Datasets to include:", [k for k, v in INCLUDE_DATASETS.items() if v])

## 2. Data Loading Functions

In [ ]:
def load_genes(genes_file: Path) -> List[str]:
    """
    Load gene names from file.
    Supports:
    - TXT/CSV with one gene per line (no header)
    - CSV with header (uses first column)
    """
    with open(genes_file, 'r') as f:
        first_line = f.readline().strip()
        f.seek(0)  # Reset to beginning
        
        # Check if first line looks like a header (contains common header names)
        if first_line.lower() in ['gene', 'genes', 'gene_name', 'gene_names', 'symbol', 'feature']:
            # Has header, skip first line
            genes = [line.strip() for line in f.readlines()[1:] if line.strip()]
        else:
            # No header, read all lines
            genes = [line.strip() for line in f if line.strip()]
    
    return genes


def load_dataset_from_mtx(
    config: Dict,
    dataset_key: str,
) -> ad.AnnData:
    """
    Load a dataset from MTX + metadata files.
    
    Parameters
    ----------
    config : dict
        Configuration dictionary for the dataset
    dataset_key : str
        Key identifier for the dataset
        
    Returns
    -------
    AnnData
        Loaded AnnData object with raw counts
    """
    print(f"\n{'='*60}")
    print(f"Loading {dataset_key}: {config['description']}")
    print(f"{'='*60}")
    
    # Load counts matrix (MTX format is genes x cells)
    print(f"  Loading counts from: {config['counts_file']}")
    counts = mmread(config['counts_file'])
    print(f"  Raw matrix shape (genes x cells): {counts.shape}")
    
    # Transpose to cells x genes for AnnData
    counts = csr_matrix(counts.T)
    print(f"  Transposed shape (cells x genes): {counts.shape}")
    
    # Load metadata
    print(f"  Loading metadata from: {config['metadata_file']}")
    metadata = pd.read_csv(config['metadata_file'], index_col=0)
    print(f"  Metadata shape: {metadata.shape}")
    
    # Load genes
    print(f"  Loading genes from: {config['genes_file']}")
    genes = load_genes(config['genes_file'])
    print(f"  Number of genes: {len(genes)}")
    
    # Verify dimensions
    assert counts.shape[0] == len(metadata), f"Cell count mismatch: {counts.shape[0]} vs {len(metadata)}"
    assert counts.shape[1] == len(genes), f"Gene count mismatch: {counts.shape[1]} vs {len(genes)}"
    
    # Create AnnData
    adata = ad.AnnData(
        X=counts,
        obs=metadata,
        var=pd.DataFrame(index=genes),
    )
    # Deduplicate gene names (pd.Index.make_unique() removed in pandas 3)
    adata.var_names_make_unique()
    
    # Add study metadata
    adata.obs['study'] = config['study_name']
    adata.obs['dataset_key'] = dataset_key
    
    # Standardize cell type column name
    if config['cell_type_col'] and config['cell_type_col'] in adata.obs.columns:
        adata.obs['cell_type'] = adata.obs[config['cell_type_col']].astype(str)
    else:
        adata.obs['cell_type'] = 'Unknown'
    
    if config.get('cell_type_detailed_col') and config['cell_type_detailed_col'] in adata.obs.columns:
        adata.obs['cell_type_detailed'] = adata.obs[config['cell_type_detailed_col']].astype(str)
    else:
        adata.obs['cell_type_detailed'] = adata.obs['cell_type']
    
    # Ensure 'Sample' column exists (use dataset_key as fallback)
    if 'Sample' not in adata.obs.columns:
        adata.obs['Sample'] = dataset_key
    
    # Create batch key combining study and sample
    adata.obs['batch'] = adata.obs['study'].astype(str) + '_' + adata.obs['Sample'].astype(str)
    
    # Basic QC stats
    print(f"\n  Summary:")
    print(f"    Cells: {adata.n_obs:,}")
    print(f"    Genes: {adata.n_vars:,}")
    print(f"    Sparsity: {(1 - adata.X.nnz / (adata.n_obs * adata.n_vars)):.1%}")
    print(f"    Cell types: {adata.obs['cell_type'].nunique()}")
    print(f"    Samples: {adata.obs['Sample'].nunique()}")
    
    return adata

## 3. Load All Datasets

In [ ]:
# Check which files exist
print("Checking data files...\n")
for key, config in DATASET_CONFIG.items():
    if INCLUDE_DATASETS.get(key, False):
        print(f"{key}:")
        for file_key in ['counts_file', 'metadata_file', 'genes_file']:
            path = config[file_key]
            exists = path.exists()
            status = "OK" if exists else "MISSING"
            print(f"  {file_key}: {path.name} [{status}]")
        print()

In [ ]:
# Load all included datasets
datasets = {}

for key, config in DATASET_CONFIG.items():
    if INCLUDE_DATASETS.get(key, False):
        # Check if all files exist
        missing_files = []
        for file_key in ['counts_file', 'metadata_file', 'genes_file']:
            if not config[file_key].exists():
                missing_files.append(config[file_key].name)
        
        if missing_files:
            print(f"\nSkipping {key}: Missing files: {missing_files}")
            continue
            
        datasets[key] = load_dataset_from_mtx(config, key)

print(f"\n{'='*60}")
print(f"Successfully loaded {len(datasets)} datasets")
print(f"{'='*60}")

In [ ]:
# Filter doublets for any dataset that has scDblFinder annotations
# (ch25h_col5a1_trem2 was pre-processed with scDblFinder in 00b notebook)
print("Doublet filtering:")
for key in list(datasets.keys()):
    adata = datasets[key]
    if 'scDblFinder.class' in adata.obs.columns:
        n_before = adata.n_obs
        datasets[key] = adata[adata.obs['scDblFinder.class'] == 'singlet'].copy()
        n_removed = n_before - datasets[key].n_obs
        print(f"  {key}: removed {n_removed:,} doublets  ({n_before:,} -> {datasets[key].n_obs:,} cells)")
    else:
        print(f"  {key}: no scDblFinder.class column, skipping")

In [ ]:
# Summary of loaded datasets
if datasets:
    summary_data = []
    for key, adata in datasets.items():
        config = DATASET_CONFIG[key]
        summary_data.append({
            'Dataset': key,
            'Study': config['study_name'],
            'Cells': f"{adata.n_obs:,}",
            'Genes': f"{adata.n_vars:,}",
            'Cell Types': adata.obs['cell_type'].nunique(),
            'Samples': adata.obs['Sample'].nunique(),
            'Conditions': adata.obs['Type'].nunique() if 'Type' in adata.obs.columns else 'N/A',
        })

    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    print(f"\nTotal cells across all datasets: {sum(a.n_obs for a in datasets.values()):,}")
else:
    print("No datasets loaded. Please check that all required files exist.")

## 4. Find Common Genes and Merge Datasets

In [ ]:
def find_common_genes(datasets: Dict[str, ad.AnnData]) -> List[str]:
    """
    Find genes common to all datasets.
    """
    gene_sets = []
    
    for key, adata in datasets.items():
        genes = set(adata.var_names)
        gene_sets.append(genes)
        print(f"{key}: {len(genes):,} genes")
    
    # Find intersection
    common_genes = gene_sets[0]
    for gs in gene_sets[1:]:
        common_genes = common_genes.intersection(gs)
    
    common_genes = sorted(list(common_genes))
    print(f"\nCommon genes across all datasets: {len(common_genes):,}")
    
    return common_genes


def merge_datasets(
    datasets: Dict[str, ad.AnnData],
    common_genes: List[str],
) -> ad.AnnData:
    """
    Merge multiple AnnData objects into one, subsetting to common genes.
    """
    adatas_to_merge = []
    
    for key, adata in datasets.items():
        print(f"Processing {key}...")
        
        # Subset to common genes
        adata_subset = adata[:, common_genes].copy()
        
        # Ensure unique cell names
        adata_subset.obs_names = [f"{key}_{i}" for i in range(adata_subset.n_obs)]
        
        adatas_to_merge.append(adata_subset)
        print(f"  Shape after subsetting: {adata_subset.shape}")
    
    # Concatenate
    print("\nConcatenating datasets...")
    adata_combined = ad.concat(adatas_to_merge, join='outer')
    
    print(f"Combined dataset shape: {adata_combined.shape}")
    
    return adata_combined


if datasets:
    common_genes = find_common_genes(datasets)
    adata_combined = merge_datasets(datasets, common_genes)
    
    print(f"\nCombined dataset shape: {adata_combined.shape}")
    print(f"\nCells per study:")
    display(adata_combined.obs['study'].value_counts())
    print(f"\nTotal batches: {adata_combined.obs['batch'].nunique()}")

In [ ]:
def preprocess_for_scvi(
    adata: ad.AnnData,
    n_top_genes: int = 3000,
    batch_key: str = 'batch',
    min_genes: int = 200,
    min_cells: int = 3,
) -> ad.AnnData:
    """
    Preprocess AnnData for scVI integration.
    """
    adata = adata.copy()
    
    # Store raw counts
    adata.layers['counts'] = adata.X.copy()
    
    # Basic filtering
    print(f"Initial shape: {adata.shape}")
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    print(f"After filtering: {adata.shape}")
    
    # Normalize for HVG selection (but keep counts in layer)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # Select highly variable genes per batch
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        batch_key=batch_key,
        flavor='seurat_v3',
        layer='counts',
        subset=False,
    )
    
    print(f"Highly variable genes: {adata.var['highly_variable'].sum()}")
    
    # Subset to HVGs
    adata = adata[:, adata.var['highly_variable']].copy()
    print(f"Final shape: {adata.shape}")
    
    return adata


if 'adata_combined' in dir():
    adata_preprocessed = preprocess_for_scvi(adata_combined, n_top_genes=3000)

In [ ]:
def preprocess_for_scvi(
    adata: ad.AnnData,
    n_top_genes: int = 3000,
    min_genes: int = 200,
    min_cells: int = 3,
) -> ad.AnnData:
    """
    Preprocess AnnData for scVI integration.

    Returns the full AnnData (all common genes) with:
      - layers['counts']          raw counts
      - X                         log1p-normalised expression
      - var['highly_variable']    HVG flag (used to subset for scVI training)

    The object is NOT subset to HVGs here so the saved .h5ad retains all
    genes for feature plots.  scVI is trained on adata[:, hvg] downstream.

    HVG selection uses batch_key='study' so every group has enough cells for
    the seurat_v3 LOESS fit (some per-sample batches have < 10 cells).
    """
    adata = adata.copy()

    # Store raw counts before any normalisation
    adata.layers['counts'] = adata.X.copy()

    # Basic filtering
    print(f"Initial shape: {adata.shape}")
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    print(f"After filtering: {adata.shape}")

    # Normalise + log1p (used for HVG selection only; model trains on 'counts')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # Mark HVGs — do NOT subset; full gene set is preserved in the object
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        batch_key='study',
        flavor='seurat_v3',
        layer='counts',
        subset=False,          # <-- keep all genes
    )

    n_hvg = adata.var['highly_variable'].sum()
    print(f"Highly variable genes selected: {n_hvg}")
    print(f"Full object shape (all genes retained): {adata.shape}")

    return adata


if 'adata_combined' in dir():
    adata_preprocessed = preprocess_for_scvi(adata_combined, n_top_genes=3000)

## 6. scVI Model Setup and Training

In [ ]:
if 'adata_preprocessed' in dir():
    # Setup scVI
    SCVI.setup_anndata(
        adata_preprocessed,
        layer='counts',
        batch_key='batch',
        categorical_covariate_keys=['study'],
    )

    print("scVI setup complete")
    print(f"Number of batches: {adata_preprocessed.obs['batch'].nunique()}")
    print(f"Number of studies: {adata_preprocessed.obs['study'].nunique()}")

In [ ]:
if 'adata_preprocessed' in dir():
    # Subset to HVGs only for scVI training.
    # The full object (adata_preprocessed) keeps all genes; adata_hvg is a
    # temporary training-only view passed to the model.
    adata_hvg = adata_preprocessed[:, adata_preprocessed.var['highly_variable']].copy()
    print(f"HVG subset for scVI training: {adata_hvg.shape}")

    SCVI.setup_anndata(
        adata_hvg,
        layer='counts',
        batch_key='batch',
        categorical_covariate_keys=['study'],
    )

    print("scVI setup complete")
    print(f"Number of batches: {adata_hvg.obs['batch'].nunique()}")
    print(f"Number of studies: {adata_hvg.obs['study'].nunique()}")

In [ ]:
if 'adata_hvg' in dir():
    model_params = {
        'n_layers': 2,
        'n_latent': 30,
        'gene_likelihood': 'nb',  # Negative binomial for count data
        'dropout_rate': 0.1,
    }

    model = SCVI(adata_hvg, **model_params)

    print(f"Model created with {model_params}")
    print(model)

In [ ]:
if 'model' in dir():
    # Train the model
    model.train(**train_params)

In [ ]:
if 'model' in dir():
    # Plot training history
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    train_history = model.history['elbo_train']
    val_history = model.history['elbo_validation']

    ax.plot(train_history.index, train_history['elbo_train'], label='Train ELBO')
    ax.plot(val_history.index, val_history['elbo_validation'], label='Validation ELBO')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ELBO')
    ax.set_title('scVI Training History')
    ax.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scvi_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Extract Latent Representation and Compute UMAP

In [ ]:
if 'model' in dir():
    # Get latent representation
    latent = model.get_latent_representation()
    adata_preprocessed.obsm['X_scvi'] = latent

    print(f"Latent representation shape: {latent.shape}")

In [ ]:
if 'model' in dir():
    # Get latent representation from the HVG-trained model and store it in
    # the full object so UMAP, Leiden, and the saved .h5ad all have all genes.
    latent = model.get_latent_representation()
    adata_preprocessed.obsm['X_scvi'] = latent

    print(f"Latent representation shape: {latent.shape}")
    print(f"Stored in adata_preprocessed (full genes: {adata_preprocessed.n_vars})")

## 8. Visualize Integration Results

In [ ]:
if 'adata_preprocessed' in dir() and 'X_umap' in adata_preprocessed.obsm:
    # UMAP overview
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sc.pl.umap(adata_preprocessed, color='study', ax=axes[0], show=False, title='Study')
    sc.pl.umap(adata_preprocessed, color='cell_type', ax=axes[1], show=False, title='Cell Type', legend_loc='right margin')
    sc.pl.umap(adata_preprocessed, color='leiden_scvi', ax=axes[2], show=False, title='Leiden Clusters')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scvi_umap_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if 'adata_preprocessed' in dir() and 'X_umap' in adata_preprocessed.obsm:
    # UMAP split by study
    studies = adata_preprocessed.obs['study'].unique()
    n_studies = len(studies)
    
    fig, axes = plt.subplots(1, n_studies, figsize=(6*n_studies, 5))
    if n_studies == 1:
        axes = [axes]

    for ax, study in zip(axes, studies):
        mask = adata_preprocessed.obs['study'] == study
        ax.scatter(
            adata_preprocessed.obsm['X_umap'][~mask, 0],
            adata_preprocessed.obsm['X_umap'][~mask, 1],
            c='lightgray', s=1, alpha=0.3
        )
        ax.scatter(
            adata_preprocessed.obsm['X_umap'][mask, 0],
            adata_preprocessed.obsm['X_umap'][mask, 1],
            c='#1f77b4', s=1, alpha=0.5
        )
        ax.set_title(study)
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scvi_umap_by_study.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if 'adata_preprocessed' in dir() and 'X_umap' in adata_preprocessed.obsm:
    # Additional metadata visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Type (condition)
    if 'Type' in adata_preprocessed.obs.columns:
        sc.pl.umap(adata_preprocessed, color='Type', ax=axes[0, 0], show=False, title='Condition/Type')
    else:
        axes[0, 0].text(0.5, 0.5, 'No Type column', ha='center', va='center')
        axes[0, 0].set_title('Condition/Type')

    # Batch
    sc.pl.umap(adata_preprocessed, color='batch', ax=axes[0, 1], show=False, title='Batch', legend_loc='none')

    # Cell type detailed
    sc.pl.umap(adata_preprocessed, color='cell_type_detailed', ax=axes[1, 0], show=False, 
               title='Cell Type (Detailed)', legend_loc='right margin')

    # Dataset key
    sc.pl.umap(adata_preprocessed, color='dataset_key', ax=axes[1, 1], show=False, title='Dataset')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scvi_umap_metadata.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Integration Quality Assessment

In [ ]:
def compute_batch_mixing(
    adata: ad.AnnData,
    batch_key: str = 'study',
    use_rep: str = 'X_scvi',
    n_neighbors: int = 50,
) -> pd.DataFrame:
    """
    Compute batch mixing score based on k-nearest neighbors.
    Higher score = better mixing.
    """
    from sklearn.neighbors import NearestNeighbors
    
    X = adata.obsm[use_rep]
    batches = adata.obs[batch_key].values
    
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(X)
    _, indices = nn.kneighbors(X)
    indices = indices[:, 1:]  # Remove self
    
    mixing_scores = []
    for cell_batch, neighbors in zip(batches, indices):
        neighbor_batches = batches[neighbors]
        mixing = (neighbor_batches != cell_batch).mean()
        mixing_scores.append(mixing)
    
    adata.obs[f'{batch_key}_mixing_score'] = mixing_scores
    
    summary = adata.obs.groupby(batch_key)[f'{batch_key}_mixing_score'].agg(['mean', 'std'])
    summary.columns = ['Mean Mixing Score', 'Std']
    
    return summary


if 'adata_preprocessed' in dir() and 'X_scvi' in adata_preprocessed.obsm:
    mixing_summary = compute_batch_mixing(adata_preprocessed, batch_key='study')
    print("Batch Mixing Scores (higher = better mixing):")
    display(mixing_summary)

In [ ]:
if 'adata_preprocessed' in dir() and 'study_mixing_score' in adata_preprocessed.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mixing score on UMAP
    sc.pl.umap(
        adata_preprocessed, 
        color='study_mixing_score', 
        ax=axes[0], 
        show=False, 
        title='Batch Mixing Score',
        cmap='RdYlGn'
    )

    # Mixing score distribution
    for study in adata_preprocessed.obs['study'].unique():
        mask = adata_preprocessed.obs['study'] == study
        scores = adata_preprocessed.obs.loc[mask, 'study_mixing_score']
        axes[1].hist(scores, alpha=0.5, label=study, bins=30)

    axes[1].set_xlabel('Mixing Score')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Mixing Score Distribution by Study')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'integration_quality.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. Save Results

In [ ]:
if 'adata_preprocessed' in dir():
    # Save integrated AnnData
    output_path = OUTPUT_DIR / 'integrated_atlas_scvi.h5ad'
    adata_preprocessed.write(output_path)
    print(f"Saved integrated atlas to: {output_path}")

if 'model' in dir():
    # Save scVI model
    model_path = OUTPUT_DIR / 'scvi_model'
    model.save(model_path, overwrite=True)
    print(f"Saved scVI model to: {model_path}")

In [ ]:
if 'adata_preprocessed' in dir() and 'model' in dir():
    import json
    
    metadata_summary = {
        'total_cells': int(adata_preprocessed.n_obs),
        'total_genes': int(adata_preprocessed.n_vars),
        'n_latent': model_params['n_latent'],
        'datasets_included': list(datasets.keys()),
        'cells_per_study': adata_preprocessed.obs['study'].value_counts().to_dict(),
        'n_leiden_clusters': int(adata_preprocessed.obs['leiden_scvi'].nunique()),
    }

    with open(OUTPUT_DIR / 'integration_metadata.json', 'w') as f:
        json.dump(metadata_summary, f, indent=2)

    print("\nIntegration Summary:")
    for k, v in metadata_summary.items():
        print(f"  {k}: {v}")

if 'adata_preprocessed' in dir() and 'model' in dir():
    import json

    metadata_summary = {
        'total_cells': int(adata_preprocessed.n_obs),
        'total_genes': int(adata_preprocessed.n_vars),        # all common genes
        'n_hvg': int(adata_preprocessed.var['highly_variable'].sum()),
        'n_latent': model_params['n_latent'],
        'datasets_included': list(datasets.keys()),
        'cells_per_study': adata_preprocessed.obs['study'].value_counts().to_dict(),
        'n_leiden_clusters': int(adata_preprocessed.obs['leiden_scvi'].nunique()),
    }

    with open(OUTPUT_DIR / 'integration_metadata.json', 'w') as f:
        json.dump(metadata_summary, f, indent=2)

    print("\nIntegration Summary:")
    for k, v in metadata_summary.items():
        print(f"  {k}: {v}")

In [ ]:
# Define canonical markers for skin cell types
SKIN_MARKERS = {
    'Keratinocytes': ['Krt14', 'Krt5', 'Krt1', 'Krt10'],
    'Fibroblasts': ['Col1a1', 'Col1a2', 'Dcn', 'Lum'],
    'Macrophages': ['Cd68', 'Adgre1', 'Csf1r', 'Mrc1'],
    'T cells': ['Cd3d', 'Cd3e', 'Cd4', 'Cd8a'],
    'Endothelial': ['Pecam1', 'Cdh5', 'Vwf', 'Kdr'],
    'Neutrophils': ['S100a8', 'S100a9', 'Ly6g', 'Cxcr2'],
    'Dendritic cells': ['Itgax', 'Cd74', 'H2-Aa', 'Flt3'],
    'Muscle': ['Acta2', 'Myh11', 'Tagln', 'Des'],
}

if 'adata_preprocessed' in dir():
    # Filter to available markers
    available_markers = {}
    for cell_type, markers in SKIN_MARKERS.items():
        present = [m for m in markers if m in adata_preprocessed.var_names]
        if present:
            available_markers[cell_type] = present

    print("Available markers:")
    for ct, markers in available_markers.items():
        print(f"  {ct}: {markers}")

In [ ]:
if 'adata_preprocessed' in dir() and 'available_markers' in dir() and available_markers:
    all_markers = [m for markers in available_markers.values() for m in markers[:2]]

    sc.pl.umap(
        adata_preprocessed, 
        color=all_markers, 
        ncols=4,
        cmap='viridis',
        use_raw=False,
    )
    plt.savefig(OUTPUT_DIR / 'marker_genes_umap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if 'adata_preprocessed' in dir() and 'available_markers' in dir() and available_markers:
    sc.pl.dotplot(
        adata_preprocessed,
        var_names=available_markers,
        groupby='leiden_scvi',
        standard_scale='var',
        dendrogram=True,
    )
    plt.savefig(OUTPUT_DIR / 'marker_genes_dotplot.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## Next Steps

The integrated atlas is now ready for downstream analysis. Proceed to `02_atlas_analysis.ipynb` for:

1. **Cell type annotation refinement** - Transfer and harmonize cell type labels
2. **Differential expression analysis** - Compare conditions (Burn vs Sham, KO vs WT)
3. **Trajectory analysis** - Study wound healing dynamics
4. **Cell type composition** - Compare cell type proportions across conditions
5. **Gene regulatory network analysis** - Identify key regulators